# Müşteri Yaşı, Demografik Özellikler ve Alışveriş Davranışlarının Harcama Üzerindeki Etkisinin İstatistiksel ve Veri Bilimi Yöntemleri ile Analizi

**Ders:** Python ile İstatistik Projesi  
**Dönem:** Bahar2026  
**Veri Seti:** Synthetic E-Commerce Dataset – Türkiye 2025  
**Kaggle:** https://www.kaggle.com/datasets/umuttuygurr/e-commerce-customer-behavior-and-sales-analysis-tr?

---

## ÖZET

Bu çalışmada, Türkiye e-ticaret pazarından elde edilen sentetik veri seti kullanılarak müşteri yaşı ve demografik özelliklerin harcama davranışları üzerindeki etkisi incelenmiştir.

**Anahtar Kelimeler:** E-ticaret, Harcama Davranışı, Yaş Grupları, İstatistiksel Analiz, Makine Öğrenmesi, Yapay Sinir Ağı

---

## ABSTRACT

In this study, the effects of customer age and demographic characteristics on spending behavior were examined using a synthetic dataset from the Turkish e-commerce market.

**Keywords:** E-commerce, Spending Behavior, Age Groups, Statistical Analysis, Machine Learning, Artificial Neural Network

In [ ]:
# ============================================================
# BÖLÜM 1: GEREKLİ KÜTÜPHANELERİN YÜKLENMESİ
# ============================================================

# Temel veri işleme
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Görselleştirme
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# İstatistiksel testler
from scipy import stats
from scipy.stats import (ttest_ind, f_oneway, chi2_contingency,
                         pearsonr, spearmanr, shapiro, levene)
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# Makine öğrenmesi
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.cluster import KMeans
from sklearn.metrics import (mean_squared_error, r2_score,
                              mean_absolute_error, silhouette_score)
from sklearn.pipeline import Pipeline

# Yapay Sinir Ağı
from sklearn.neural_network import MLPRegressor

# TensorFlow/Keras (derin öğrenme)
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
    from tensorflow.keras.optimizers import Adam
    TF_AVAILABLE = True
    print(f"TensorFlow {tf.__version__} yüklendi.")
except ImportError:
    TF_AVAILABLE = False
    print("TensorFlow bulunamadı – SKLearn MLPRegressor kullanılacak.")

# Görsel stil ayarları
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#F8F9FA',
    'axes.grid': True,
    'grid.alpha': 0.4,
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
})
sns.set_palette('husl')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ===== GLOBAL: Age labels (kullanılan tüm bölümlerde erişilebilir) =====
AGE_LABELS = ['18-25', '26-35', '36-45', '46-60', '60+']
AGE_PALETTE = ['#E74C3C', '#E67E22', '#2ECC71', '#3498DB', '#9B59B6']

print("Tüm kütüphaneler başarıyla yüklendi!")
print(f"Pandas {pd.__version__} | NumPy {np.__version__}")

In [ ]:
# ============================================================
# BÖLÜM 2: VERİ SETİ YÜKLEME
# ============================================================

import os, glob

# --- Dosya yolu: aynı klasördeki ilk CSV dosyasını otomatik bul ---
csv_files = glob.glob('./*.csv') + glob.glob('../*.csv') + glob.glob('./data/*.csv')

if csv_files:
    DATA_PATH = csv_files[0]
    print(f"Bulunan dosya: {DATA_PATH}")
    df_raw = pd.read_csv(DATA_PATH, encoding='utf-8-sig')
else:
    raise FileNotFoundError(
        "CSV dosyası bulunamadı!\n"
        "Kaggle'dan 'synthetic-e-commerce-dataset-trkiye-2025' dosyasını indirip "
        "bu notebook ile aynı klasöre koyun."
    )

print(f"\nVeri seti boyutu: {df_raw.shape[0]:,} satır × {df_raw.shape[1]} sütun")
print("\nSütun adları:")
for i, col in enumerate(df_raw.columns, 1):
    print(f"  {i:2}. {col}  ({df_raw[col].dtype})")

In [ ]:
# ============================================================
# SÜTUN HARİTALAMA – Kaggle veri setindeki sütun adlarını
# standart değişken isimlerine eşleştir
# ============================================================

COLUMN_MAP_TR = {
    # Müşteri bilgileri
    'müşteri_id':           'musteri_id',
    'musteri_id':           'musteri_id',
    'customer_id':          'musteri_id',
    'MüşteriID':            'musteri_id',

    'yaş':                  'yas',
    'yas':                  'yas',
    'age':                  'yas',
    'Age':                  'yas',
    'Yaş':                  'yas',

    'cinsiyet':             'cinsiyet',
    'gender':               'cinsiyet',
    'Gender':               'cinsiyet',
    'Cinsiyet':             'cinsiyet',

    'şehir':                'sehir',
    'sehir':                'sehir',
    'city':                 'sehir',
    'City':                 'sehir',
    'Şehir':                'sehir',

    'bölge':                'bolge',
    'region':               'bolge',

    'ürün_kategorisi':      'kategori',
    'urun_kategorisi':      'kategori',
    'kategori':             'kategori',
    'category':             'kategori',
    'Category':             'kategori',
    'product_category':     'kategori',

    'ödeme_yöntemi':        'odeme_yontemi',
    'odeme_yontemi':        'odeme_yontemi',
    'payment_method':       'odeme_yontemi',
    'PaymentMethod':        'odeme_yontemi',

    'kampanya':             'kampanya',
    'campaign':             'kampanya',
    'Campaign':             'kampanya',
    'kampanya_durumu':      'kampanya',
    'is_campaign':          'kampanya',
    'discount_applied':     'kampanya',

    'sipariş_miktarı':      'siparis_miktari',
    'siparis_miktari':      'siparis_miktari',
    'quantity':             'siparis_miktari',
    'Quantity':             'siparis_miktari',
    'adet':                 'siparis_miktari',

    'birim_fiyat':          'birim_fiyat',
    'unit_price':           'birim_fiyat',
    'UnitPrice':            'birim_fiyat',
    'fiyat':                'birim_fiyat',
    'price':                'birim_fiyat',

    # HEDEF DEĞİŞKEN: Toplam Harcama
    'toplam_harcama':       'toplam_harcama',
    'total_spending':       'toplam_harcama',
    'TotalSpending':        'toplam_harcama',
    'harcama':              'toplam_harcama',
    'spending':             'toplam_harcama',
    'total_amount':         'toplam_harcama',
    'TotalAmount':          'toplam_harcama',
    'amount':               'toplam_harcama',
    'siparis_tutari':       'toplam_harcama',
    'sipariş_tutarı':       'toplam_harcama',
    'order_amount':         'toplam_harcama',
    'sales':                'toplam_harcama',
    'Sales':                'toplam_harcama',
    'revenue':              'toplam_harcama',
    'toplam_tutar':         'toplam_harcama',

    'tarih':                'tarih',
    'date':                 'tarih',
    'Date':                 'tarih',
    'siparis_tarihi':       'tarih',
    'order_date':           'tarih',

    'musteri_puani':        'musteri_puani',
    'customer_score':       'musteri_puani',
    'loyalty_score':        'musteri_puani',
    'rating':               'musteri_puani',
    'Rating':               'musteri_puani',

    'iade':                 'iade',
    'return':               'iade',
    'is_returned':          'iade',
    'returned':             'iade',
}

# Sütun adlarını küçük harfe çevir ve boşlukları alt çizgiye dönüştür
df = df_raw.copy()
df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]

# Haritalama uygula
rename_dict = {}
for orig_col in df.columns:
    if orig_col in COLUMN_MAP_TR:
        rename_dict[orig_col] = COLUMN_MAP_TR[orig_col]

df = df.rename(columns=rename_dict)

print("Standardize edilmiş sütun adları:")
print(df.columns.tolist())

# --- Hedef değişken kontrolü ---
if 'toplam_harcama' not in df.columns:
    # FIX #1: En BÜYÜK ORTALAMA değere sahip sütunu seç (max single value değil)
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    candidate = df[numeric_cols].mean().idxmax()
    df = df.rename(columns={candidate: 'toplam_harcama'})
    print(f"\nUYARI: 'toplam_harcama' sütunu bulunamadı. '{candidate}' hedef değişken olarak atandı.")
    print("Gerekirse COLUMN_MAP_TR sözlüğüne kendi sütun adınızı ekleyin.")

print(f"\nHedef değişken (Y): toplam_harcama")
print(df.head(3))

In [ ]:
# İlk genel bakış
print("=" * 60)
print("VERİ SETİ GENEL BİLGİLERİ")
print("=" * 60)
df.info()

print("\n" + "=" * 60)
print("BETİMSEL İSTATİSTİKLER")
print("=" * 60)
display(df.describe(include='all').T)

---
## 5. VERİ TEMİZLEME VE ÖN İŞLEME

In [ ]:
# ============================================================
# BÖLÜM 3: EKSİK VERİ ANALİZİ
# ============================================================

print("EKSİK VERİ ANALİZİ")
print("-" * 50)
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Eksik Sayı': missing, 'Eksik Oran (%)': missing_pct})
missing_df = missing_df[missing_df['Eksik Sayı'] > 0].sort_values('Eksik Oran (%)', ascending=False)

if len(missing_df) == 0:
    print("Veri setinde eksik değer bulunmamaktadır. ✓")
else:
    print(missing_df)

# Eksik değer görselleştirme
fig, ax = plt.subplots(figsize=(10, 4))
if len(missing_df) > 0:
    missing_df['Eksik Oran (%)'].plot(kind='bar', color='#E74C3C', ax=ax)
    ax.set_title('Sütunlara Göre Eksik Veri Oranı (%)', fontsize=14, fontweight='bold')
    ax.set_xlabel('Sütun')
    ax.set_ylabel('Eksik Oran (%)')
    for p in ax.patches:
        ax.annotate(f'{p.get_height():.1f}%', (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='bottom', fontsize=10)
else:
    ax.text(0.5, 0.5, 'Eksik Değer Yok ✓', ha='center', va='center',
            fontsize=16, color='green', transform=ax.transAxes)
    ax.set_title('Eksik Veri Analizi', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('grafik_01_eksik_veri.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nYorum: Eksik değerler toplam veri setinin %5'ini aşmıyorsa medyan/mod ile doldurulur;")

In [ ]:
# ============================================================
# BÖLÜM 4: EKSİK DEĞERLERİ DOLDURMA
# ============================================================

# Sayısal sütunlar → medyan ile doldur
numeric_cols = df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        # FIX #8: inplace parameter yerine assignment kullan (deprecation warning)
        df[col] = df[col].fillna(df[col].median())
        print(f"  {col}: eksik değerler medyan ({df[col].median():.2f}) ile dolduruldu.")

# Kategorik sütunlar → mod ile doldur
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        mode_val = df[col].mode()[0]
        df[col] = df[col].fillna(mode_val)
        print(f"  {col}: eksik değerler mod ('{mode_val}') ile dolduruldu.")

print(f"\nVeri temizleme sonrası eksik değer: {df.isnull().sum().sum()}")
print(f"Veri seti boyutu: {df.shape}")

In [ ]:
# ============================================================
# BÖLÜM 5: AYKIRI DEĞER ANALİZİ (IQR Yöntemi)
# ============================================================

def detect_outliers_iqr(series, name):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = series[(series < lower) | (series > upper)]
    print(f"  {name:25s} | Q1={Q1:8.2f} | Q3={Q3:8.2f} | IQR={IQR:8.2f} "
          f"| Alt Sınır={lower:8.2f} | Üst Sınır={upper:8.2f} "
          f"| Aykırı Sayısı={len(outliers)} ({len(outliers)/len(series)*100:.1f}%)")
    return lower, upper

print("AYKIRI DEĞER ANALİZİ (IQR Yöntemi)")
print("-" * 100)
bounds = {}
for col in numeric_cols:
    if col in df.columns:
        lo, hi = detect_outliers_iqr(df[col], col)
        bounds[col] = (lo, hi)

# Aykırı değer görselleştirme – Boxplot
plot_cols = [c for c in ['toplam_harcama', 'yas', 'birim_fiyat', 'siparis_miktari', 'musteri_puani']
             if c in df.columns]

if plot_cols:
    fig, axes = plt.subplots(1, len(plot_cols), figsize=(5 * len(plot_cols), 5))
    if len(plot_cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, plot_cols):
        ax.boxplot(df[col].dropna(), patch_artist=True,
                   boxprops=dict(facecolor='#3498DB', alpha=0.7),
                   medianprops=dict(color='#E74C3C', linewidth=2))
        ax.set_title(col.replace('_', ' ').title(), fontweight='bold')
        ax.set_ylabel('Değer')
    plt.suptitle('Sayısal Değişkenler için Boxplot – Aykırı Değer Analizi',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('grafik_02_boxplot_aykiri.png', dpi=150, bbox_inches='tight')
    plt.show()

# Winsorization ile aykırı değerleri sınırlama (silmek yerine)
if 'toplam_harcama' in df.columns and 'toplam_harcama' in bounds:
    lo, hi = bounds['toplam_harcama']
    df['toplam_harcama'] = df['toplam_harcama'].clip(lower=lo, upper=hi)
    print(f"\nWinsorization uygulandı: toplam_harcama [{lo:.2f}, {hi:.2f}] aralığına sınırlandı.")

print("\nYorum: Winsorization, aykırı değerleri silmek yerine sınır değerlerine eşitler.")

---
## 6. YAŞ GRUPLAMALARI

In [ ]:
# ============================================================
# BÖLÜM 6: YAŞ GRUPLAMA
# ============================================================

if 'yas' in df.columns:
    # FIX #2: Doğru bin aralıkları (0-25 yerine 17-25)
    bins   = [17, 25, 35, 45, 60, 120]
    df['yas_grubu'] = pd.cut(df['yas'], bins=bins, labels=AGE_LABELS, right=True)

    print("Yaş Grubu Dağılımı:")
    dist = df['yas_grubu'].value_counts().sort_index()
    print(dist)
    print(f"\nOrtalama Yaş: {df['yas'].mean():.1f}")
    print(f"Medyan Yaş  : {df['yas'].median():.1f}")
    print(f"Min Yaş     : {df['yas'].min()}")
    print(f"Max Yaş     : {df['yas'].max()}")

    # Yaş grubu + ortalama harcama tablosu
    print("\nYaş Grubuna Göre Ortalama Harcama:")
    print(df.groupby('yas_grubu', observed=True)['toplam_harcama'].agg(['mean','median','std','count']).round(2))
else:
    print("UYARI: 'yas' sütunu bulunamadı. Sütun haritalamasını kontrol edin.")

---
## 7. KEŞİFÇİ VERİ ANALİZİ (EDA)

In [ ]:
# ============================================================
# BÖLÜM 7A: HEDEF DEĞİŞKEN DAĞILIMI
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df['toplam_harcama'].dropna(), bins=40, color='#2ECC71', edgecolor='white', alpha=0.85)
axes[0].axvline(df['toplam_harcama'].mean(), color='#E74C3C', linestyle='--', linewidth=2,
                label=f"Ortalama: {df['toplam_harcama'].mean():.0f}")
axes[0].axvline(df['toplam_harcama'].median(), color='#3498DB', linestyle='-.', linewidth=2,
                label=f"Medyan: {df['toplam_harcama'].median():.0f}")
axes[0].set_title('Toplam Harcama Dağılımı (Histogram)', fontweight='bold')
axes[0].set_xlabel('Toplam Harcama (TL)')
axes[0].set_ylabel('Frekans')
axes[0].legend()

# KDE (Yoğunluk Eğrisi)
df['toplam_harcama'].dropna().plot(kind='kde', ax=axes[1], color='#9B59B6', linewidth=2.5)
axes[1].set_title('Toplam Harcama Yoğunluk Eğrisi (KDE)', fontweight='bold')
axes[1].set_xlabel('Toplam Harcama (TL)')
axes[1].set_ylabel('Yoğunluk')

plt.suptitle('Hedef Değişken: Toplam Harcama Dağılım Analizi', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('grafik_03_harcama_dagilim.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Çarpıklık (Skewness): {df['toplam_harcama'].skew():.3f}")
print(f"Basıklık (Kurtosis) : {df['toplam_harcama'].kurtosis():.3f}")
print("\nYorum: Skewness > 1 ise dağılım sağa çarpık.")

In [ ]:
# ============================================================
# BÖLÜM 7B: YAŞ GRUBUNA GÖRE HARCAMA – BOXPLOT
# ============================================================

if 'yas_grubu' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # FIX #3 & #4: AGE_LABELS ve AGE_PALETTE global değişkenlerini kullan
    # Boxplot
    sns.boxplot(data=df, x='yas_grubu', y='toplam_harcama', palette=AGE_PALETTE,
                order=AGE_LABELS, ax=axes[0])
    axes[0].set_title('Yaş Grubuna Göre Harcama Dağılımı', fontweight='bold')
    axes[0].set_xlabel('Yaş Grubu')
    axes[0].set_ylabel('Toplam Harcama (TL)')

    # Violin
    sns.violinplot(data=df, x='yas_grubu', y='toplam_harcama', palette=AGE_PALETTE,
                   order=AGE_LABELS, ax=axes[1], inner='quartile')
    axes[1].set_title('Yaş Grubuna Göre Harcama Yoğunluğu (Violin)', fontweight='bold')
    axes[1].set_xlabel('Yaş Grubu')
    axes[1].set_ylabel('Toplam Harcama (TL)')

    plt.suptitle('Yaş Grubu × Toplam Harcama İlişkisi', fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('grafik_04_yas_harcama_boxplot.png', dpi=150, bbox_inches='tight')
    plt.show()

    print("Yorum: Medyan çizgisinin konumu yaş grupları arasındaki merkezi eğilimi gösterir.")

In [ ]:
# ============================================================
# BÖLÜM 7C: KATEGORİ VE CİNSİYET GRAFİKLERİ
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1) Kategori bazında ortalama harcama
if 'kategori' in df.columns:
    cat_mean = df.groupby('kategori')['toplam_harcama'].mean().sort_values(ascending=False)
    bars = axes[0, 0].bar(cat_mean.index, cat_mean.values,
                          color=sns.color_palette('husl', len(cat_mean)))
    axes[0, 0].set_title('Ürün Kategorisine Göre Ortalama Harcama', fontweight='bold')
    axes[0, 0].set_xlabel('Kategori')
    axes[0, 0].set_ylabel('Ortalama Harcama (TL)')
    axes[0, 0].tick_params(axis='x', rotation=45)
    for bar in bars:
        axes[0, 0].text(bar.get_x() + bar.get_width() / 2.,
                        bar.get_height() + 5, f'{bar.get_height():.0f}',
                        ha='center', va='bottom', fontsize=9)

# 2) Cinsiyet dağılımı pie chart
if 'cinsiyet' in df.columns:
    gender_counts = df['cinsiyet'].value_counts()
    axes[0, 1].pie(gender_counts.values, labels=gender_counts.index,
                   autopct='%1.1f%%', colors=['#3498DB', '#E74C3C', '#2ECC71'],
                   startangle=90, shadow=True)
    axes[0, 1].set_title('Cinsiyet Dağılımı', fontweight='bold')

# 3) Cinsiyet × Harcama
if 'cinsiyet' in df.columns:
    sns.barplot(data=df, x='cinsiyet', y='toplam_harcama',
                palette=['#3498DB', '#E74C3C', '#2ECC71'],
                estimator=np.mean, errorbar='sd', ax=axes[1, 0])
    axes[1, 0].set_title('Cinsiyete Göre Ortalama Harcama (±SD)', fontweight='bold')
    axes[1, 0].set_xlabel('Cinsiyet')
    axes[1, 0].set_ylabel('Ortalama Harcama (TL)')

# 4) Ödeme yöntemi countplot
if 'odeme_yontemi' in df.columns:
    pay_counts = df['odeme_yontemi'].value_counts()
    axes[1, 1].bar(pay_counts.index, pay_counts.values,
                   color=sns.color_palette('Set2', len(pay_counts)))
    axes[1, 1].set_title('Ödeme Yöntemi Dağılımı', fontweight='bold')
    axes[1, 1].set_xlabel('Ödeme Yöntemi')
    axes[1, 1].set_ylabel('İşlem Sayısı')
    axes[1, 1].tick_params(axis='x', rotation=45)

plt.suptitle('Demografik ve Davranışsal Değişkenlerin Görselleştirilmesi',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('grafik_05_demografik_analiz.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# BÖLÜM 7D: KORELASYON ISI HARİTASI
# ============================================================

numeric_df = df.select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr()

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, vmin=-1, vmax=1,
            linewidths=0.5, square=True, ax=ax,
            annot_kws={'size': 10})
ax.set_title('Sayısal Değişkenler Korelasyon Matrisi (Alt Üçgen)',
             fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('grafik_06_korelasyon_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

if 'toplam_harcama' in corr_matrix.columns:
    print("Toplam Harcama ile Korelasyonlar (|r| > 0.1):")
    target_corr = corr_matrix['toplam_harcama'].drop('toplam_harcama').abs().sort_values(ascending=False)
    for col, val in target_corr.items():
        if val > 0.1:
            print(f"  {col:30s}: r = {corr_matrix['toplam_harcama'][col]:.3f}")

In [ ]:
# ============================================================
# BÖLÜM 7E: PAIRPLOT (EN ÖNEMLI DEĞİŞKENLER)
# ============================================================

pair_cols = [c for c in ['yas', 'toplam_harcama', 'birim_fiyat', 'siparis_miktari', 'musteri_puani']
             if c in df.columns]

hue_col = 'yas_grubu' if 'yas_grubu' in df.columns else ('cinsiyet' if 'cinsiyet' in df.columns else None)

if len(pair_cols) >= 2:
    # FIX #7: Duplicate column check
    cols_to_use = pair_cols + ([hue_col] if hue_col and hue_col not in pair_cols else [])
    pp = sns.pairplot(df[cols_to_use].dropna(),
                      hue=hue_col, plot_kws={'alpha': 0.4, 's': 20},
                      diag_kind='kde', corner=True)
    pp.figure.suptitle('Seçili Değişkenler Arası İlişki (Pairplot)',
                       fontsize=14, fontweight='bold', y=1.01)
    plt.savefig('grafik_07_pairplot.png', dpi=120, bbox_inches='tight')
    plt.show()
    print("Yorum: Köşegen üzerindeki KDE eğrileri her değişkenin dağılımını gösterir.")

In [ ]:
# ============================================================
# BÖLÜM 7F: YAŞ GRUBUNA GÖRE KATEGORİ ISISI (HEATMAP)
# ============================================================

if 'yas_grubu' in df.columns and 'kategori' in df.columns:
    pivot = df.pivot_table(values='toplam_harcama', index='yas_grubu',
                           columns='kategori', aggfunc='mean', observed=True)

    fig, ax = plt.subplots(figsize=(14, 5))
    sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlOrRd',
                linewidths=0.5, ax=ax, cbar_kws={'label': 'Ortalama Harcama (TL)'})
    ax.set_title('Yaş Grubu × Ürün Kategorisine Göre Ortalama Harcama (TL)',
                 fontsize=14, fontweight='bold')
    ax.set_xlabel('Ürün Kategorisi')
    ax.set_ylabel('Yaş Grubu')
    plt.tight_layout()
    plt.savefig('grafik_08_yas_kategori_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Yorum: Koyu renkler yüksek ortalama harcamayı gösterir.")

---
## 8. BETİMSEL İSTATİSTİK

In [ ]:
# ============================================================
# BÖLÜM 8: BETİMSEL İSTATİSTİK
# ============================================================

def descriptive_stats(series, name):
    """Kapsamlı betimsel istatistik hesaplar."""
    s = series.dropna()
    stats_dict = {
        'Değişken'      : name,
        'N'             : len(s),
        'Ortalama'      : round(s.mean(), 3),
        'Medyan'        : round(s.median(), 3),
        # FIX #6: Mode hesaplamasında float() dönüşümü ekle
        'Mod'           : round(float(s.mode().iloc[0]), 3) if not s.mode().empty else 'N/A',
        'Std Sapma'     : round(s.std(), 3),
        'Varyans'       : round(s.var(), 3),
        'Min'           : round(s.min(), 3),
        'Max'           : round(s.max(), 3),
        'Q1 (25%)'      : round(s.quantile(0.25), 3),
        'Q3 (75%)'      : round(s.quantile(0.75), 3),
        'IQR'           : round(s.quantile(0.75) - s.quantile(0.25), 3),
        'Çarpıklık'     : round(s.skew(), 3),
        'Basıklık'      : round(s.kurtosis(), 3),
        'Var. Katsayısı': round(s.std() / s.mean() * 100, 1) if s.mean() != 0 else 'N/A',
    }
    return stats_dict

stat_cols = [c for c in ['toplam_harcama', 'yas', 'birim_fiyat',
                          'siparis_miktari', 'musteri_puani'] if c in df.columns]

stats_list = [descriptive_stats(df[col], col) for col in stat_cols]
stats_table = pd.DataFrame(stats_list).set_index('Değişken')

print("BETİMSEL İSTATİSTİK TABLOSU")
print("=" * 80)
display(stats_table)

# Yaş grubuna göre betimsel istatistik
if 'yas_grubu' in df.columns:
    print("\nYAŞ GRUBUNA GÖRE HARCAMA İSTATİSTİKLERİ")
    print("=" * 80)
    yas_stats = df.groupby('yas_grubu', observed=True)['toplam_harcama'].agg([
        ('N', 'count'),
        ('Ortalama', 'mean'),
        ('Medyan', 'median'),
        ('Std Sapma', 'std'),
        ('Min', 'min'),
        ('Max', 'max'),
    ]).round(2)
    display(yas_stats)

---
## 9. HİPOTEZ TESTLERİ

In [ ]:
# ============================================================
# BÖLÜM 9A: NORMALLİK TESTİ (Shapiro-Wilk)
# ============================================================

print("NORMALLİK TESTİ (Shapiro-Wilk)")
print("-" * 50)
print("H0: Veri normal dağılmaktadır.")
print("H1: Veri normal dağılmamaktadır.")
print()

# Shapiro-Wilk en fazla 5000 örnek için güvenilirdir
sample = df['toplam_harcama'].dropna().sample(min(5000, len(df)), random_state=RANDOM_STATE)
stat_sw, p_sw = shapiro(sample)
print(f"İstatistik : {stat_sw:.4f}")
print(f"p-değeri   : {p_sw:.6f}")
print()
if p_sw < 0.05:
    print("Karar: H0 REDDEDİLDİ – Veri normal dağılmamaktadır (p < 0.05).")
    print("→ Parametrik testlerde örneklem büyüklüğü yeterince büyük olduğu için")
    print("  Merkezi Limit Teoremi gereği t-testi ve ANOVA uygulanabilir.")
else:
    print("Karar: H0 REDDEDİLEMEDİ – Veri normal dağılmaktadır (p ≥ 0.05).")

In [ ]:
# ============================================================
# BÖLÜM 9B: BAĞIMSIZ ÖRNEKLEM t-TESTİ
# ============================================================

if 'yas_grubu' in df.columns:
    genc = df[df['yas_grubu'].isin(['18-25', '26-35'])]['toplam_harcama'].dropna()
    olgun = df[df['yas_grubu'].isin(['36-45', '46-60', '60+'])]['toplam_harcama'].dropna()

    print("BAĞIMSIZ ÖRNEKLEM t-TESTİ")
    print("-" * 60)
    print("H0: Genç ve olgun müşterilerin ortalama harcaması eşittir (μ₁ = μ₂)")
    print("H1: Farklıdır (μ₁ ≠ μ₂)")
    print(f"\nGenç  (18-35): n={len(genc):,}, ort.={genc.mean():.2f} TL, std={genc.std():.2f}")
    print(f"Olgun (36+)  : n={len(olgun):,}, ort.={olgun.mean():.2f} TL, std={olgun.std():.2f}")

    # Levene varyans homojenlik testi
    lev_stat, lev_p = levene(genc, olgun)
    equal_var = lev_p >= 0.05
    print(f"\nLevene Testi: p={lev_p:.4f} → "
          f"{'Varyanslar eşit (Student t-testi)' if equal_var else 'Varyanslar farklı (Welch t-testi)'}")

    t_stat, p_val = ttest_ind(genc, olgun, equal_var=equal_var)
    print(f"\nt-istatistiği : {t_stat:.4f}")
    print(f"p-değeri      : {p_val:.6f}")
    alpha = 0.05
    print(f"\nKarar (α={alpha}): H0 {'REDDEDİLDİ ✗ – Anlamlı fark var' if p_val < alpha else 'REDDEDİLEMEDİ ✓ – Anlamlı fark yok'}")

    # Cohen's d etki büyüklüğü
    pooled_std = np.sqrt((genc.std()**2 + olgun.std()**2) / 2)
    cohens_d = (genc.mean() - olgun.mean()) / pooled_std
    print(f"Cohen's d     : {cohens_d:.3f} "
          f"({'Küçük' if abs(cohens_d) < 0.2 else 'Orta' if abs(cohens_d) < 0.8 else 'Büyük'} etki)")

In [ ]:
# ============================================================
# BÖLÜM 9C: TEK YÖNLÜ ANOVA
# ============================================================

if 'yas_grubu' in df.columns:
    print("TEK YÖNLÜ ANOVA")
    print("-" * 60)
    print("H0: Tüm yaş gruplarının ortalama harcaması eşittir (μ₁=μ₂=μ₃=μ₄=μ₅)")
    print("H1: En az bir yaş grubunun ortalaması farklıdır.")

    # FIX #5: Sadece mevcut yaş gruplarını getir, sırayla eşle
    existing_groups = [g for g in AGE_LABELS if g in df['yas_grubu'].values]
    groups = [df[df['yas_grubu'] == g]['toplam_harcama'].dropna().values
              for g in existing_groups]

    f_stat, p_anova = f_oneway(*groups)
    print(f"\nF-istatistiği : {f_stat:.4f}")
    print(f"p-değeri      : {p_anova:.6f}")
    print(f"\nKarar: H0 {'REDDEDİLDİ ✗ – Yaş grupları arasında anlamlı fark var' if p_anova < 0.05 else 'REDDEDİLEMEDİ ✓ – Anlamlı fark yok'}")

    # Eta-kare etki büyüklüğü
    grand_mean = df['toplam_harcama'].mean()
    ss_between = sum(len(g) * (np.mean(g) - grand_mean)**2 for g in groups)
    ss_total = sum((df['toplam_harcama'].dropna() - grand_mean)**2)
    eta_sq = ss_between / ss_total
    print(f"Eta-kare (η²) : {eta_sq:.4f} "
          f"({'Küçük' if eta_sq < 0.06 else 'Orta' if eta_sq < 0.14 else 'Büyük'} etki)")

    # POST-HOC: Tukey HSD
    if p_anova < 0.05:
        print("\nPOST-HOC ANALİZİ: Tukey HSD")
        print("(Hangi gruplar birbirinden farklı?)")
        tukey_data = df[['yas_grubu', 'toplam_harcama']].dropna()
        tukey = pairwise_tukeyhsd(tukey_data['toplam_harcama'], tukey_data['yas_grubu'], alpha=0.05)
        print(tukey)

    # ANOVA görselleştirme
    fig, ax = plt.subplots(figsize=(12, 5))
    group_means = [np.mean(g) for g in groups]
    group_sds   = [np.std(g) for g in groups]
    # FIX #5: Correct label-to-group mapping
    bars = ax.bar(existing_groups, group_means, yerr=group_sds,
                  capsize=6, color=AGE_PALETTE[:len(existing_groups)], alpha=0.85, edgecolor='white')
    ax.axhline(grand_mean, color='black', linestyle='--', linewidth=1.5, label=f'Genel Ort: {grand_mean:.0f}')
    for bar, mean in zip(bars, group_means):
        ax.text(bar.get_x() + bar.get_width() / 2., bar.get_height() + 10,
                f'{mean:.0f}', ha='center', va='bottom', fontweight='bold')
    ax.set_title(f'Yaş Grubuna Göre Ortalama Harcama (ANOVA: F={f_stat:.2f}, p={p_anova:.4f})',
                 fontweight='bold')
    ax.set_xlabel('Yaş Grubu')
    ax.set_ylabel('Ortalama Toplam Harcama (TL)')
    ax.legend()
    plt.tight_layout()
    plt.savefig('grafik_09_anova.png', dpi=150, bbox_inches='tight')
    plt.show()